# Dropout
은닉층의 뉴런이 많아질수록 모델은 복잡한 패턴을 더 잘 표현할 수 있다.
하지만 그만큼 학습 데이터의 우연한 잡음까지 외워버릴 가능성도 있다.

Dropout은 학습 중 일부 뉴런을 확률적으로 꺼서, 특정 뉴런 몇 개에만 지나치게 의존하지 않도록 만드는 대표적인 regularization 기법이다.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)

## 1. Dropout 동작 방법
- 학습시에 일부 뉴런이 무작위로 비활성화 된다.
- 평가 시에는 모든 뉴런을 사용한다.
- 따라서 'model.train()', 'model.eval()'의 전환이 매우 중요하다.

In [ ]:
x = torch.tensor([1., 2., 3., 4., 5.])

# Dropout 객체 생성 (각 원소를 약 40%의 확률로 끈다)
drop = nn.Dropout(p=0.4)

# 학습 모드 : 일부 값이 랜덤하게 0이 된다.
# 살아 남은 값은 1 / (1 - p) 만큼 키워서 출력한다. (dropout의 스케일 보정)
drop.train()
print('train mode 1:', drop(x))
print('train mode 2:', drop(x))
print('train mode 3:', drop(x))

# 평가 모드 : dropout을 저용하지 않고 입력을 그대로 사용한다.
drop.eval()
print('eval mode 1:', drop(x))
print('eval mode 2:', drop(x))
print('eval mode 3:', drop(x))

train mode 1: tensor([1.6667, 3.3333, 5.0000, 6.6667, 8.3333])
train mode 2: tensor([1.6667, 0.0000, 0.0000, 6.6667, 8.3333])
train mode 3: tensor([1.6667, 3.3333, 0.0000, 0.0000, 8.3333])


## 2. dropout rate 'p'의 의미
보통 너무 작으면 regularization 효과가 약하고, 너무 크면 학습에 필요한 정보까지 과하게 잃을 수 있다.
실무적으로 완전한 규칙이 있는 것은 아니지만, MLP에서는 '0.2 ~ 0.5 부근을 자주 실험해본다.

In [ ]:
# 같은 입력에 대해 p값이 다를 때의 예시
x = torch.ones(10)

for p in [0.1, 0.3, 0.5, 0.7]:
    drop = nn.Dropout(p=p)
    drop.train()
    print(f'p={p}:', {drop(x)})

- 선형 변환을 하고
- 활성화 함수를 통과 시킨 뒤
- 그 결과 일부를 학습 중에만 무작위로 끄는 식이다.

In [ ]:
class MLPwithoutDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [ ]:
# 모델 학습 함수
def train_binary_model(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.01):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):
        # 학습 모드 : dropout이 켜진다.
        model.train()
        optimizer.zero_grad()
        train_logits = model(X_train)
        train_loss = criterion(train_logits, y_train)
        optimizer.step()

        with torch.no_grad():
            train_pred = (torch.sigmoid(train_logits) > 0.5).float()
            train_acc = (train_pred == y_train).float().mean().item()

        # 평가 모드 : dropout이 꺼진다.
        model.eval()
        with torch.no_grad():
                val_logits = model(X_val)
                val_loss = criterion(val_logits, y_val) 
                val_pred = (torch.sigmoid(val_logits) > 0.5).float()
                val_acc = (val_pred == y_val).float().mean().item()

        train_losses.append(train_loss.item())
        val_losses.append(val_loss.item())
        train_accs.append(train_acc)
        val_accs.append(val_acc)
    
    return train_losses, val_losses, train_accs, val_accs

In [ ]:
torch.manual_seed(42)
model_no_dropout = MLPwithoutDropout()
torch.manual_seed(42)
model_dropout = MLPwithDropout()

hist_no = train_binary_model(model_no_dropout, X_train, y_train, X_val, y_val)
hist_do= train_binary_model(model_dropout, X_train, y_train, X_val, y_val)

no_train_losses, no_val_losses, no_train_acc, no_val_acc = hist_no
do_train_losses, do_val_losses, do_train_acc, do_val_acc = hist_do

print('without dropout - final train acc:', round(no_train_acc[-1],4))
print('without dropout - final val acc:', round(no_val_acc[-1],4))

print('with dropout - final train acc:', round(do_train_acc[-1],4))
print('with dropout - final val acc:', round(do_val_acc[-1],4))

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(no_train_losses, label='train loss - no dropout')
plt.plot(no_val_losses, label='val loss - no dropout')
plt.plot(do_train_losses, label='train loss - dropout')
plt.plot(do_val_losses, label='val loss - dropout')

plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

1. Dropout이 없는 모델은 훈련 성능이 더 빨리 좋아질 수 있다.
2. 하지만 검증 성능이 비슷하거나 오히려 불안정해질 수 있다.
3. Dropout이 있는 모델은 훈련이 약간 더 어렵지만, 일반화 측면에서 이점이 생길 수 있다.